# Google Colab Host for EduMind AI Backend & Ollama

Run this cell to setup Ollama, pull the LLM model, configure and start the FastAPI backend, and expose it via Cloudflare Tunnel.

In [ ]:
# =====================================================================
# 1. Clone the GitHub Repository & Switch to Committee Branch
# =====================================================================
import os
import getpass

repo_url = "github.com/jaynishthakar/demo-.git"
project_dir = "demo-"
branch_name = "commitee-process-and-fixes"

if not os.path.exists(project_dir):
    print("Cloning repository...")
    is_private = input("Is the GitHub repository private? (yes/no): ").strip().lower() == "yes"
    if is_private:
        token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ").strip()
        clone_url = f"https://{token}@{repo_url}"
    else:
        clone_url = f"https://{repo_url}"

    !git clone -b {branch_name} {clone_url}
else:
    print("Repository already exists. Pulling latest updates...")
    %cd {project_dir}
    !git checkout {branch_name}
    !git pull
    %cd ..

%cd {project_dir}

# =====================================================================
# 2. Install Python & System Dependencies
# =====================================================================
print("\nInstalling system dependencies (zstd)...")
!apt-get update && apt-get install -y zstd

print("\nInstalling python packages (requirements.txt)...")
!pip install -r requirements.txt

# =====================================================================
# 3. Install & Start Ollama (CORS Enabled)
# =====================================================================
print("\nInstalling Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("\nStarting Ollama service in background...")
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print("\nPulling Qwen2.5:7b model (takes ~1-2 mins)...")
!ollama pull qwen2.5:7b

# =====================================================================
# 4. Start the FastAPI Backend
# =====================================================================
print("\nStarting FastAPI Backend in the background...")
backend_env = os.environ.copy()
backend_env["OLLAMA_BASE_URL"] = "http://localhost:11434"
backend_env["LLM_BACKEND"] = "ollama"
backend_env["OLLAMA_MODEL"] = "qwen2.5:7b"

backend_log = open("backend_server.log", "w")
subprocess.Popen(
    ["python", "-m", "uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"],
    env=backend_env, stdout=backend_log, stderr=backend_log
)
time.sleep(8)

# =====================================================================
# 5. Install & Start Cloudflare Tunnel on Backend Port (8000)
# =====================================================================
print("\nInstalling Cloudflare Tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("\nStarting Cloudflare Tunnel on Backend Port 8000...")
!pkill cloudflared

with open("cloudflare_tunnel.log", "w") as log_file:
    subprocess.Popen([
        "cloudflared", "tunnel", "--protocol", "http2",
        "--url", "http://localhost:8000"
    ], stdout=log_file, stderr=log_file)

time.sleep(10)

# =====================================================================
# 6. Retrieve and Print the Live Backend URL
# =====================================================================
import re
try:
    with open("cloudflare_tunnel.log", "r") as log_file:
        log_content = log_file.read()
        urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
        if urls:
            print("\n" + "="*70)
            print(" SUCCESS: BACKEND IS NOW ONLINE")
            print("-"*70)
            print(" Copy this URL into your Vercel Frontend UI:")
            print(f" {urls[0]}")
            print("="*70 + "\n")
        else:
            print("\nTunnel URL not found in log yet. Run '!cat cloudflare_tunnel.log' in a new cell in 5 seconds.")
except Exception as e:
    print(f"Error starting tunnel: {e}")
